In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import os

In [4]:
loader = PyPDFLoader("../data/data_science_syllabus.pdf")
docs = loader.load()
len(docs)

10

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

10

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-2-preview")

In [7]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [8]:
query = "Machine Learning and Data Science Content"
data = vector_store.similarity_search(query=query)

In [9]:
context = ""
for doc in data:
    context += doc.page_content + "\n"

In [10]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.environ.get("GROQ_API_KEY"))

In [11]:
res = llm.invoke(f"can you provide me answer based on provided context for anu questions, context: {context} and question: {query} " )

In [ ]:
#print(res.content)

Below is a concise, **placement‑ready** summary of the **Machine‑Learning & Data‑Science** content that appears in the 1‑Year Roadmap you shared.  
It pulls together every module, topic, tool, and hands‑on project that a learner will encounter, so you can quickly answer any “what do you study in ML/DS?” type of question.

---

## 1️⃣ Overview – Where ML/DS Fits in the Year‑Long Path  

| Phase | Core Focus | Typical Duration | Why it matters for a placement‑ready profile |
|------|------------|------------------|---------------------------------------------|
| **Module 6 – ML I (Supervised Learning)** | Foundations of predictive modeling | **Month 6 (4 weeks)** | Gives you the end‑to‑end pipeline that every Data Scientist or ML Engineer must master. |
| **Module 7 – ML II (Unsupervised & Feature Engineering)** | Clustering, dimensionality reduction, data‑prep tricks | **Month 7 (3 weeks)** | Shows you how to explore data, discover patterns, and prepare clean features – a must for any a

In [13]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [15]:
from langchain_core.prompts import PromptTemplate

In [16]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

In [17]:
rag_chain = get_context | prompt | llm

In [18]:
res = rag_chain.invoke("What is the value of RBC Count and is it in range ? ")

In [19]:
print(res.content)

I don’t know.
